In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# TODO: Load the CSV file into df

df_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(df_path)

print(f"Shape: {df.shape}")

In [ ]:
# Task 2: Write your code here:
#Inspect the first few rows using head()
df.head()

In [ ]:
# Task 3: Write your code here:
#Display dataset information using info()
df.info()

In [ ]:
# Task 4: Write your code here:
#Show statistical description using describe()
df.describe()

In [ ]:
# Task 5: Write your code here:
#Plot the target distribution (delivery_time)
print("\nValue counts for the target variable 'class':")
class_distribution = df['Delivery_Time'].value_counts()
print(class_distribution)

# Delivery Time distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df_clean = df.copy() #ALWAYS COPY

In [ ]:
# Task 1: Write your code here:
#Drop the 'Order_ID' column from the data
df_clean=df_clean.drop(columns=['Order_ID'])

In [ ]:
df_clean.info()

In [ ]:
# Task 2: Write your code here:
#Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :)
df_clean = df_clean.dropna(subset=['Delivery_Time']) #target cannot be empty

# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna('unknown')

# Fill Courier_Experience_yrs  with mode - discrete feature, mode is most representative
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mean())



In [ ]:
# Task 3: Write your code here:
#Check and remove duplicates if any exist
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True) #edit on og df
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# 3. Do we have categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {} #Creates an empty dictionary
for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])
  label_encoders[col] = le #This line stores the fitted LabelEncoder for each categorical column
                           #so the same encoding can be reused on new data or reversed later.

df_clean

In [ ]:
from sklearn.preprocessing import StandardScaler
# Task 5: Write your code here:
#Apply feature scaling for all features (Use StandardScaler)
#selecting numerical columns
# ["int64", "float64"] == ["number"]
#numerical_cols = df_clean.select_dtypes(include=["number"],exclude=["object"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

#scaler = StandardScaler()
#df_clean[numerical_cols] = scaler.fit_transform(df[numerical_cols])
#df_clean.head()


In [ ]:
import seaborn as sns

# Task 6: Write your code here:
#Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True)) #each value ÷ total number of rows (between 0 and 1 )
                                                        #(normalize=False) will show true numbers
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_clean, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
#Split the dataset into features (X) and target (y)
# we use .astype(float) because some models expect ONLY float and not integers
X = df.drop("Delivery_Time", axis=1)
y = df['Delivery_Time'].astype(float)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
# sigmoid in NumPy
def sigmoid(z):
  return 1 / (1 + np.exp(-z))

def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape # m rows, n columns (dimensions)
  theta = np.zeros(n) # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Logistic Regression"):
    z = np.dot(X, theta)
    y_hat = sigmoid(z)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = binary_cross_entropy(y, y_hat)
    losses.append(loss)

  return theta, losses

%matplotlib inline
# Task 2,3,4,5: Write your code here:
#Use the correct split: KFold OR StratifiedKFold
#Train a RandomForest model
#Evaluate using MAE (Mean Absolute Error) ONLY
#Print the averaged score across all folds
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Storage for logistic regression results for each fold
lr_losses = []
lr_accuracy = []
lr_precision = []
lr_recall = []
lr_f1 = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
    theta, losses = gradient_descent(X_train, y_train, learning_rate=0.5, n_iters=500)

    # Validate
    y_pred_proba = sigmoid(np.dot(X_test, theta))
    y_pred = (y_pred_proba >= 0.5).astype(int) #if we increse 0.5 to 0.8

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Store results
    lr_losses.append(losses)
    lr_accuracy.append(accuracy)
    lr_precision.append(precision)
    lr_recall.append(recall)
    lr_f1.append(f1)

    # Calculate average loss across folds
avg_loss = np.mean(lr_losses, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(avg_loss, label='Logistic Regression Loss', color='purple')
plt.title('Average Logistic Regression Loss Curve (Across 5 Folds)')
plt.xlabel('Iteration')
plt.ylabel('Binary Cross-Entropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
# Calculate average loss across folds
avg_loss = np.mean(lr_losses, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(avg_loss, label='Logistic Regression Loss', color='purple')
plt.title('Average Logistic Regression Loss Curve (Across 5 Folds)')
plt.xlabel('Iteration')
plt.ylabel('Binary Cross-Entropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
sklearn_models = {

  "Random Forest": RandomForestClassifier(
      n_estimators=320,  # Number of trees
      max_depth=4
  )
}
all_results = {}

for name in sklearn_models:
  all_results[name] = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}


n_splits = 5 # K

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)


for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  for model_name, model in sklearn_models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train) # train
    y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    all_results[model_name]['accuracy'].append(accuracy)
    all_results[model_name]['precision'].append(precision)
    all_results[model_name]['recall'].append(recall)
    all_results[model_name]['f1'].append(f1)

    for model_name in all_results:
      print(f"\n{model_name}:")
      print(f"  Accuracy:  {np.mean(all_results[model_name]['accuracy']):.4f}")
      print(f"  Precision: {np.mean(all_results[model_name]['precision']):.4f}")
      print(f"  Recall:    {np.mean(all_results[model_name]['recall']):.4f}")
      print(f"  F1-Score:  {np.mean(all_results[model_name]['f1']):.4f}")

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: